# Strands Agent with Langfuse Observability on Amazon Bedrock AgentCore Runtime

## Overview

This notebook demonstrates deploying a Strands agent to Amazon Bedrock AgentCore Runtime with Langfuse observability integration. The implementation uses Amazon Bedrock Claude models and sends telemetry data to Langfuse through OpenTelemetry (OTEL).

## Key Components

- **Strands Agents**: Python framework for building LLM-powered agents with built-in telemetry support
- **Amazon Bedrock AgentCore Runtime**: Managed runtime service for hosting and scaling agents on AWS
- **Langfuse**: Open-source observability platform for LLM applications that receives traces via OTEL
- **OpenTelemetry**: Industry-standard protocol for collecting and exporting telemetry data

## Architecture

The agent is containerized and deployed to AgentCore Runtime, which provides HTTP endpoints for invocation. Telemetry data flows from the Strands agent through OTEL exporters to Langfuse for monitoring and debugging. The implementation disables AgentCore's default observability to use Langfuse instead.

## Prerequisites

- Python 3.10+
- AWS credentials configured with Bedrock and AgentCore permissions
- [Langfuse](https://langfuse.com/) account with API keys (public and secret keys)
- Docker installed locally
- Access to Amazon Bedrock Claude models in us-west-2

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet
!pip install --force-reinstall -U -r requirements-dev.txt --quiet

## Configure AWS Credentials

## Agent Implementation

The agent file (`strands_claude.py`) implements a travel agent with web search capabilities. Key configuration includes:
- Initializing Strands telemetry

In [1]:
%%writefile strands_claude.py
import os
import logging
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent, tool
from strands.models import BedrockModel
from strands.telemetry import StrandsTelemetry
from ddgs import DDGS

logging.basicConfig(level=logging.ERROR, format="[%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)
logger.setLevel(os.getenv("AGENT_RUNTIME_LOG_LEVEL", "INFO").upper())


@tool
def web_search(query: str) -> str:
    """
    Search the web for information using DuckDuckGo.

    Args:
        query: The search query

    Returns:
        A string containing the search results
    """
    try:
        ddgs = DDGS()
        results = ddgs.text(query, max_results=5)

        formatted_results = []
        for i, result in enumerate(results, 1):
            formatted_results.append(
                f"{i}. {result.get('title', 'No title')}\n"
                f"   {result.get('body', 'No summary')}\n"
                f"   Source: {result.get('href', 'No URL')}\n"
            )

        return "\n".join(formatted_results) if formatted_results else "No results found."

    except Exception as e:
        return f"Error searching the web: {str(e)}"

# Function to initialize Bedrock model
def get_bedrock_model():
    region = os.getenv("AWS_DEFAULT_REGION", "us-west-2")
    model_id = os.getenv("BEDROCK_MODEL_ID", "us.anthropic.claude-3-7-sonnet-20250219-v1:0")

    bedrock_model = BedrockModel(
        model_id=model_id,
        region_name=region,
        temperature=0.0,
        max_tokens=1024
    )
    return bedrock_model

# Initialize the Bedrock model
bedrock_model = get_bedrock_model()

# Define the agent's system prompt
system_prompt = """You are an experienced travel agent specializing in personalized travel recommendations 
with access to real-time web information. Your role is to find dream destinations matching user preferences 
using web search for current information. You should provide comprehensive recommendations with current 
information, brief descriptions, and practical travel details."""

app = BedrockAgentCoreApp()

def initialize_agent():
    """Initialize the agent with proper telemetry configuration."""

    # Initialize Strands telemetry with 3P configuration
    strands_telemetry = StrandsTelemetry()
    strands_telemetry.setup_otlp_exporter()
    
    # Create and cache the agent
    agent = Agent(
        model=bedrock_model,
        system_prompt=system_prompt,
        tools=[web_search]
    )
    
    return agent

@app.entrypoint
def strands_agent_bedrock(payload, context=None):
    """
    Invoke the agent with a payload
    """
    user_input = payload.get("prompt")
    logger.info("[%s] User input: %s", context.session_id, user_input)
    
    # Initialize agent with proper configuration
    agent = initialize_agent()
    
    response = agent(user_input)
    return response.message['content'][0]['text']

if __name__ == "__main__":
    app.run()

Writing strands_claude.py


### Configure AgentCore Runtime deployment

Next we will use our starter toolkit to configure the AgentCore Runtime deployment with an entrypoint, the execution role we just created and a requirements file. We will also configure the starter kit to auto create the Amazon ECR repository on launch.

During the configure step, your docker file will be generated based on your application code. Please note that when using the `bedrock_agentcore_starter_toolkit` to configure your agent, it configures AgentCore Observability by default so, to use Braintrust, you need to remove configuration for AgentCore Observability as explained below:

<div style="text-align:left">
    <img src="../images/configure.png" width="40%"/>
</div>

In [3]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()
agent_name = "strands_langfuse_observability"
response = agentcore_runtime.configure(
    entrypoint="strands_claude.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,
    memory_mode='NO_MEMORY',
    disable_otel=True,
)
response

Entrypoint parsed: file=/Users/ravizraj/Documents/workspaces/agent-core/rajesh/amazon-bedrock-agentcore-samples/01-tutorials/06-AgentCore-observability/04-Agentcore-runtime-partner-observability/Langfuse/strands_claude.py, bedrock_agentcore_name=strands_claude
Memory disabled - agent will be stateless
Configuring BedrockAgentCore agent: strands_langfuse_observability
Memory disabled
Generated Dockerfile: Dockerfile
Generated .dockerignore: /Users/ravizraj/Documents/workspaces/agent-core/rajesh/amazon-bedrock-agentcore-samples/01-tutorials/06-AgentCore-observability/04-Agentcore-runtime-partner-observability/Langfuse/.dockerignore
Keeping 'strands_langfuse_observability' as default agent
Bedrock AgentCore configured: /Users/ravizraj/Documents/workspaces/agent-core/rajesh/amazon-bedrock-agentcore-samples/01-tutorials/06-AgentCore-observability/04-Agentcore-runtime-partner-observability/Langfuse/.bedrock_agentcore.yaml


ConfigureResult(config_path=PosixPath('/Users/ravizraj/Documents/workspaces/agent-core/rajesh/amazon-bedrock-agentcore-samples/01-tutorials/06-AgentCore-observability/04-Agentcore-runtime-partner-observability/Langfuse/.bedrock_agentcore.yaml'), dockerfile_path=PosixPath('/Users/ravizraj/Documents/workspaces/agent-core/rajesh/amazon-bedrock-agentcore-samples/01-tutorials/06-AgentCore-observability/04-Agentcore-runtime-partner-observability/Langfuse/Dockerfile'), dockerignore_path=PosixPath('/Users/ravizraj/Documents/workspaces/agent-core/rajesh/amazon-bedrock-agentcore-samples/01-tutorials/06-AgentCore-observability/04-Agentcore-runtime-partner-observability/Langfuse/.dockerignore'), runtime='Finch', region='us-east-1', account_id='467801433859', execution_role=None, ecr_repository=None, auto_create_ecr=True, memory_id=None)

## Deploy to AgentCore Runtime

Now that we've got a docker file, let's launch the agent to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime

<div style="text-align:left">
    <img src="../images/launch.png" width="75%"/>
</div>

In [4]:
import base64
from dotenv import dotenv_values

config = dotenv_values(".env")

# Langfuse configuration
otel_endpoint = config.get("LANGFUSE_OTEL_ENDPOINT", "https://us.cloud.langfuse.com/api/public/otel")
langfuse_secret_key = config.get("LANGFUSE_SECRET_KEY", "")  # For production key should be securely stored
langfuse_public_key = config.get("LANGFUSE_PUBLIC_KEY", "")  # For production key should be securely stored
langfuse_auth_token = base64.b64encode(f"{langfuse_public_key}:{langfuse_secret_key}".encode()).decode()
otel_auth_header = f"Authorization=Basic {langfuse_auth_token}"

# Bedrock configuration
model_id = config.get("BEDROCK_MODEL_ID", "us.anthropic.claude-3-7-sonnet-20250219-v1:0")

# # Langfuse configuration
# otel_endpoint = "https://us.cloud.langfuse.com/api/public/otel"
# langfuse_secret_key = "<langfuse-secret-key>"  # For production key should be securely stored
# langfuse_public_key = "<langfuse-private-key>"  # For production key should be securely stored
# langfuse_auth_token = base64.b64encode(f"{langfuse_public_key}:{langfuse_secret_key}".encode()).decode()
# otel_auth_header = f"Authorization=Basic {langfuse_auth_token}"


launch_result = agentcore_runtime.launch(
    env_vars={
        "BEDROCK_MODEL_ID": model_id, # Example model ID
        "OTEL_EXPORTER_OTLP_ENDPOINT": otel_endpoint,  # Use Langfuse OTEL endpoint
        "OTEL_EXPORTER_OTLP_HEADERS": otel_auth_header,  # Add Langfuse OTEL auth header
        "DISABLE_ADOT_OBSERVABILITY": "true",
    }
)
launch_result


🚀 CodeBuild mode: building in cloud (RECOMMENDED - DEFAULT)
   • Build ARM64 containers in the cloud with CodeBuild
   • No local Docker required
💡 Available deployment modes:
   • runtime.launch()                           → CodeBuild (current)
   • runtime.launch(local=True)                 → Local development
   • runtime.launch(local_build=True)           → Local build + cloud deploy (NEW)
Memory disabled - skipping memory creation
Starting CodeBuild ARM64 deployment for agent 'strands_langfuse_observability' to account 467801433859 (us-east-1)
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: strands_langfuse_observability


Repository doesn't exist, creating new ECR repository: bedrock-agentcore-strands_langfuse_observability


✅ ECR repository available: 467801433859.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-strands_langfuse_observability
Getting or creating execution role for agent: strands_langfuse_observability
Using AWS region: us-east-1, account ID: 467801433859
Role name: AmazonBedrockAgentCoreSDKRuntime-us-east-1-2102f51ebb
Role doesn't exist, creating new execution role: AmazonBedrockAgentCoreSDKRuntime-us-east-1-2102f51ebb
Starting execution role creation process for agent: strands_langfuse_observability
✓ Role creating: AmazonBedrockAgentCoreSDKRuntime-us-east-1-2102f51ebb
Creating IAM role: AmazonBedrockAgentCoreSDKRuntime-us-east-1-2102f51ebb
✓ Role created: arn:aws:iam::467801433859:role/AmazonBedrockAgentCoreSDKRuntime-us-east-1-2102f51ebb
✓ Execution policy attached: BedrockAgentCoreRuntimeExecutionPolicy-strands_langfuse_observability
Role creation complete and ready for use with Bedrock AgentCore
✅ Execution role available: arn:aws:iam::467801433859:role/AmazonBedrockAgentCoreSDKRunt

LaunchResult(mode='codebuild', tag='bedrock_agentcore-strands_langfuse_observability:latest', env_vars=None, port=None, runtime=None, ecr_uri='467801433859.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-strands_langfuse_observability', agent_id='strands_langfuse_observability-Dpdabr6vpf', agent_arn='arn:aws:bedrock-agentcore:us-east-1:467801433859:runtime/strands_langfuse_observability-Dpdabr6vpf', codebuild_id='bedrock-agentcore-strands_langfuse_observability-builder:840922f5-fdf5-474a-bff0-a7c2f2908669', build_output=None)

## Check Deployment Status

Wait for the runtime to be ready before invoking:

In [5]:
import time
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(status)
status

Retrieved Bedrock AgentCore status for: strands_langfuse_observability


'READY'

### Invoking AgentCore Runtime

Finally, we can invoke our AgentCore Runtime with a payload

<div style="text-align:left">
    <img src="../images/invoke.png" width=75%"/>
</div>

In [6]:
invoke_response = agentcore_runtime.invoke({"prompt": "I'm planning a weekend trip to london. What are the must-visit places and local food I should try?"})

In [7]:
from IPython.display import Markdown, display
display(Markdown("".join(invoke_response['response'])))

"# London Weekend Trip: Must-Visit Places & Local Food\n\nBased on the latest information, here's a comprehensive guide for your weekend trip to London:\n\n## Must-Visit Places\n\n### Iconic Landmarks\n1. **Tower of London** - Historic castle and former royal residence housing the Crown Jewels\n2. **London Eye** - Iconic observation wheel offering panoramic views of the city\n3. **Big Ben & Houses of Parliament** - Famous clock tower and the UK's government buildings\n4. **Buckingham Palace** - The Queen's official London residence (check for Changing of the Guard ceremony times)\n5. **Tower Bridge** - Iconic Victorian bridge with glass walkways and engine rooms to explore\n\n### Museums & Galleries (Many are free!)\n1. **British Museum** - World-class collection spanning human history and culture\n2. **Tate Modern** - Contemporary art in a converted power station\n3. **Natural History Museum** - Dinosaurs, wildlife exhibits, and the famous blue whale skeleton\n4. **Victoria & Albert Museum (V&A)** - Art and design museum with incredible collections\n\n### Markets & Neighborhoods\n1. **Borough Market** - London's oldest food market, perfect for sampling local treats\n2. **Notting Hill** - Charming neighborhood with colorful houses and Portobello Road Market\n3. **Covent Garden** - Shopping, street performers, and the Royal Opera House\n4. **South Bank** - Riverside walk with cultural venues, street food, and great views\n\n### Parks & Green Spaces\n1. **Hyde Park** - Vast royal park perfect for a leisurely stroll\n2. **Crystal Palace Park** - Mentioned as a must-visit free attraction\n\n## Local Food to Try\n\n### Traditional British Dishes\n1. **Fish and Chips** - Battered fish with thick-cut fries, a British comfort food classic\n2. **Pie and Mash** - Traditional meat pies served with mashed potatoes and parsley sauce (\"liquor\")\n3. **Sunday Roast** - Roasted meat with Yorkshire pudding, vegetables, and gravy (best enjoyed in a traditional pub)\n4. **Full English Breakfast** - Eggs, bacon, sausage, beans, toast, and more\n5. **Scotch Eggs** - Hard-boiled egg wrapped in sausage meat and breadcrumbs, then deep-fried\n\n### London Specialties\n1. **Afternoon Tea** - An elegant tradition with finger sandwiches, scones, and pastries\n2. **Chicken Tikka Masala** - Britain's adopted national dish, best tried in Brick Lane\n3. **Jellied Eels** - A traditional East End specialty (for the adventurous!)\n4. **Craft Beer** - London has a thriving craft beer scene with many local breweries\n\n## Practical Tips\n- Consider getting an Oyster card or using contactless payment for public transportation\n- Many museums are free, so you can save money while seeing world-class collections\n- For efficient sightseeing, group attractions by area (e.g., Tower of London and Tower Bridge)\n- Borough Market is perfect for lunch while exploring central London\n- The Sky Garden offers free panoramic views (but requires advance booking)\n- Consider a West End show for evening entertainment\n\nWould you like more specific information about any of these attractions or food recommendations? Or do you have any other questions about your London weekend trip?"

### Invoking AgentCore Runtime with boto3

Now that your AgentCore Runtime was created you can invoke it with any AWS SDK. For instance, you can use the boto3 `invoke_agent_runtime` method for it.

In [10]:
import boto3
import yaml
import json
from IPython.display import Markdown, display

# Read agent manifest
with open('.bedrock_agentcore.yaml', 'r') as f:
    agent_manifest = yaml.safe_load(f)

agent_name = agent_manifest["default_agent"]
region = agent_manifest["agents"][agent_name]["aws"]["region"]
agent_arn = agent_manifest["agents"][agent_name]["bedrock_agentcore"]["agent_arn"]

agentcore_client = boto3.client(
    'bedrock-agentcore',
    region_name=region
)

boto3_response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    qualifier="DEFAULT",
    payload=json.dumps({"prompt": "What is 2+2?"})
)
if "text/event-stream" in boto3_response.get("contentType", ""):
    content = []
    for line in boto3_response["response"].iter_lines(chunk_size=1):
        if line:
            line = line.decode("utf-8")
            if line.startswith("data: "):
                line = line[6:]
                print(line)
                content.append(line)
    display(Markdown("\n".join(content)))
else:
    try:
        events = []
        for event in boto3_response.get("response", []):
            events.append(event)
    except Exception as e:
        events = [f"Error reading EventStream: {e}"]
    display(Markdown(json.loads(events[0].decode("utf-8"))))

I notice your question is about a simple math calculation rather than travel recommendations. 

The answer to 2+2 is 4.

If you're interested in travel recommendations, I'd be happy to help you find dream destinations based on your preferences. Just let me know what kind of travel experience you're looking for, such as beach destinations, cultural experiences, adventure travel, or any specific regions you're interested in visiting.

## View Traces in Langfuse

To view the traces:
1. Go to your Langfuse dashboard at https://cloud.langfuse.com
2. Navigate to your project
3. Click on "Traces" to view the telemetry data

The traces will include:
- Agent invocation details
- Tool calls (web search)
- Model interactions with latency and token usage
- Request/response payloads

## Cleanup (Optional)

Clean up the deployed resources:

In [ ]:
import boto3
import yaml

# Read agent manifest
with open('.bedrock_agentcore.yaml', 'r') as f:
    agent_manifest = yaml.safe_load(f)

# print(json.dumps(agent_manifest, indent=4))
agent_name = agent_manifest["default_agent"]
region = agent_manifest["agents"][agent_name]["aws"]["region"]
agent_id = agent_manifest["agents"][agent_name]["bedrock_agentcore"]["agent_id"]
repository_name = agent_manifest["agents"][agent_name]["aws"]["ecr_repository"].split('/')[1]
codebuild_project_name = agent_manifest["agents"][agent_name]["codebuild"]["project_name"]
ac_role = agent_manifest["agents"][agent_name]["aws"]["execution_role"]
cb_role = agent_manifest["agents"][agent_name]["codebuild"]["execution_role"]
s3_bucket = agent_manifest["agents"][agent_name]["codebuild"]["source_bucket"]

region, agent_id, repository_name, codebuild_project_name

In [ ]:
agentcore_control_client = boto3.client(
    'bedrock-agentcore-control',
    region_name=region
)
delete_runtime_response = agentcore_control_client.delete_agent_runtime(
    agentRuntimeId=agent_id
)

ecr_client = boto3.client(
    'ecr',
    region_name=region
)
delete_ecr_response = ecr_client.delete_repository(
    repositoryName=repository_name,
    force=True
)

codebuild_client = boto3.client(
    'codebuild',
    region_name=region
)
delete_codebuild_response = codebuild_client.delete_project(
    name=codebuild_project_name
)

print(f"Delete AgentCore Runtime Execution IAM Role: {ac_role}")
print(f"Delete Code Build Execution IAM Role: {cb_role}")
print(f"S3 Artifacts folder: s3://{s3_bucket}/{agent_name}/")

## Summary

You have successfully deployed a Strands agent to Amazon Bedrock AgentCore Runtime with Langfuse observability. The implementation demonstrates:
- Integration of Strands agents with AgentCore Runtime
- Configuration of OpenTelemetry to send traces to Langfuse
- Proper initialization order to ensure telemetry configuration
- Invocation through both SDK and boto3 client

The agent is now running in a managed, scalable environment with full observability through Langfuse.